In [1]:
import pandas as pd
import numpy as np
import itertools
import os
df = pd.read_csv(r"C:\Users\Samuel\FH_Aachen\tsib_simuflex_fork\tsib_simuflex\tsib\data\episcope\episcope.csv")

In [2]:
def episcope_combis(path: str, country: str = "DE", building_types: list = ["SFH", "MFH", "TH", "AB"])-> dict:
    df = pd.read_csv(path, delimiter=",")
    df = df.copy()
    df = df[(df["Code_Country"] == country) & (df["Code_BuildingSizeClass"].isin(building_types)) & (df["Code_DataType_Building"] == "ReEx") & (df["Code_BuildingVariant"].str.contains(".N.", regex=False))]
    cons = {
        "B_N2": "Terraced",
        "B_N1": "Semi",
        "B_Alone": "Detached"
    }
    df["Connections"] = df["Code_AttachedNeighbours"].map(cons)

    test = df[["Code_BuildingSizeClass", "Code_ConstructionYearClass", "Connections"]].drop_duplicates()
    #print(test.index)
    tzu = df.loc[test.index]
    tzu = tzu[["Code_BuildingSizeClass","Connections", "Code_ConstructionYearClass"]]
    tzu.columns = ["Type", "Surrounding","AgeBin"]
    res_dict = tzu.to_dict("records")
    return res_dict
episcope_combis(r"C:\Users\Samuel\FH_Aachen\tsib_simuflex_fork\tsib_simuflex\tsib\data\episcope\episcope.csv")

[{'Type': 'AB', 'Surrounding': 'Terraced', 'AgeBin': 'DE.02'},
 {'Type': 'AB', 'Surrounding': 'Semi', 'AgeBin': 'DE.03'},
 {'Type': 'AB', 'Surrounding': 'Detached', 'AgeBin': 'DE.04'},
 {'Type': 'AB', 'Surrounding': 'Detached', 'AgeBin': 'DE.05'},
 {'Type': 'AB', 'Surrounding': 'Detached', 'AgeBin': 'DE.06'},
 {'Type': 'MFH', 'Surrounding': 'Detached', 'AgeBin': 'DE.01'},
 {'Type': 'MFH', 'Surrounding': 'Terraced', 'AgeBin': 'DE.02'},
 {'Type': 'MFH', 'Surrounding': 'Detached', 'AgeBin': 'DE.03'},
 {'Type': 'MFH', 'Surrounding': 'Detached', 'AgeBin': 'DE.04'},
 {'Type': 'MFH', 'Surrounding': 'Semi', 'AgeBin': 'DE.05'},
 {'Type': 'MFH', 'Surrounding': 'Detached', 'AgeBin': 'DE.06'},
 {'Type': 'MFH', 'Surrounding': 'Detached', 'AgeBin': 'DE.07'},
 {'Type': 'MFH', 'Surrounding': 'Detached', 'AgeBin': 'DE.08'},
 {'Type': 'MFH', 'Surrounding': 'Semi', 'AgeBin': 'DE.09'},
 {'Type': 'MFH', 'Surrounding': 'Detached', 'AgeBin': 'DE.10'},
 {'Type': 'MFH', 'Surrounding': 'Detached', 'AgeBin': 'DE

In [3]:
len(episcope_combis(r"C:\Users\Samuel\FH_Aachen\tsib_simuflex_fork\tsib_simuflex\tsib\data\episcope\episcope.csv"))

40

In [4]:
#building type
#n apartments
#occ
# flat area



In [5]:
#https://episcope.eu/building-typology/country/de/  Statistics of the German Building Stock Source[2]
def gen_flat_count(buildingtype: str, rng: np.random.Generator) -> int:
    if buildingtype == "SFH":
        return 1
    elif buildingtype == "TH":
        return int(rng.integers(low=1, high=3))
    elif buildingtype == "MFH":
        return int(rng.integers(low=3, high=13)) #exclusive
    elif buildingtype == "AB":
        return int(rng.integers(low=13, high=21))
    else:
        raise ValueError

In [6]:
#https://www.destatis.de/DE/Themen/Gesellschaft-Umwelt/Wohnen/Tabellen/tabelle-wo2-mietwohnungen.html
#https://www.destatis.de/DE/Themen/Gesellschaft-Umwelt/Wohnen/Tabellen/tabelle-wo2-eigentuemerwohnungen.html
#probs sind für haushalte
def gen_occs(buildingtype: str, apartment_count: int, rng: np.random.Generator) ->list:

    occ_probs = {
        "SFH": (0.245114, 0.402323, 0.154148, 0.148869, 0.049623),
        "MFH": (0.490622, 0.307419, 0.101949, 0.074417, 0.024805),
        "AB": (0.490622, 0.307419, 0.101949, 0.074417, 0.024805),
        "TH": (0.236817, 0.400092, 0.157261, 0.154371, 0.051457),
    }
    probs = occ_probs[buildingtype]

    num_occs =[]

    for _ in range(apartment_count):
        draw = rng.random()
        x = 0.0
        for i,v in enumerate(probs, start=1):
            x += v
            if x > draw:
                num_occs.append(i)
                break
        else:
            num_occs.append(len(probs))



    return num_occs

In [7]:
#https://www.destatis.de/DE/Themen/Gesellschaft-Umwelt/Wohnen/Tabellen/tabelle-wo4-wohnflaeche.html
def gen_flat_area(occs: list ,rng: np.random.Generator)-> list:
    area_occ_probs = {
        1: {
            "area": [(20, 39), (40, 59), (60, 79), (80, 99), (100, 119), (120, 139), (140, 200)],
            "probs": [0.09605,0.30524,0.27605,0.13575,0.07434,0.05302, 0.05955]
        },
        2: {
            "area": [(20, 39), (40, 59), (60, 79), (80, 99), (100, 119), (120, 139), (140, 200)],
            "probs": [0.00677,0.08684,0.23555,0.19991,0.15465,0.13864, 0.17764]
        },
        3: {
            "area": [(20, 39), (40, 59), (60, 79), (80, 99), (100, 119), (120, 139), (140, 200)],
            "probs": [0.0,0.03780,0.19027,0.20132,0.15757,0.16097, 0.25207]
        },
        4: {
            "area": [(20, 39), (40, 59), (60, 79), (80, 99), (100, 119), (120, 139), (140, 200)],
            "probs": [0.0,0.01389,0.11820,0.17035,0.14979,0.17815, 0.36962]
        },
    }
    areas = []

    for occ in occs:
        if occ > 4:
            occ = 4

        res = rng.choice(area_occ_probs[occ]["area"], p=area_occ_probs[occ]["probs"])
        floor = rng.uniform(low=res[0],high= res[1])
        areas.append(floor)

    return areas

# Manifest Generator

In [8]:

def manifest(scenario_reps,seed_reps,episcope_path , buildingtypes: list = ["SFH", "MFH", "TH", "AB"] , seed=42):
    combinations = episcope_combis(episcope_path, building_types= buildingtypes)
    year_types = ["average"] #"hot", "cold"
    future = [False] #True
    rng = np.random.default_rng(seed)
    rows = []
    scenario_id = 0

    grid  = itertools.product(combinations, year_types, future)
    for x,y,z in grid:
        b_type = x["Type"]

        for _ in range(scenario_reps):
            n_flats = gen_flat_count(buildingtype=b_type, rng=rng)
            occs = gen_occs(buildingtype=b_type,apartment_count=n_flats, rng=rng)
            flat_areas = gen_flat_area(occs=occs,rng=rng)
            total_area = sum(flat_areas)
            clima_region = int(rng.integers(low=1, high=16))

            for _ in range(seed_reps):
                rows.append(dict(
                    country = "DE",
                    buildingType = b_type,
                    surrounding = x["Surrounding"],
                    buildingAgeBin = x["AgeBin"],
                    a_ref = round(total_area,1),
                    n_apartments = n_flats,
                    year_type = y,
                    future = z,
                    climateRegion = clima_region,
                    n_persons = occs,
                    freq = "1min",
                    hasFirePlace = False,
                    cores = 1,
                    scenario_id = scenario_id
                )
                    )
            scenario_id += 1
    manifest = pd.DataFrame(rows)
    manifest["run_id"] = manifest.index.map(lambda i: f"run_{i:06d}")
    manifest["seed"]= 1000000 + manifest.index
    manifest.to_parquet(r"manifest.parquet")
    return manifest



manifest(1,1,r"C:\Users\Samuel\FH_Aachen\tsib_simuflex_fork\tsib_simuflex\tsib\data\episcope\episcope.csv", seed=45)


,country,buildingType,surrounding,buildingAgeBin,a_ref,n_apartments,year_type,future,climateRegion,n_persons,freq,hasFirePlace,cores,scenario_id,run_id,seed
0,DE,AB,Terraced,DE.02,1938.3,20,average,False,9,"[2, 2, 3, 2, 2, 2, 2, 1, 2, 2, 3, 2, 2, 4, 1, ...",1min,False,1,0,run_000000,1000000
1,DE,AB,Semi,DE.03,1363.8,16,average,False,14,"[1, 3, 2, 1, 3, 2, 1, 2, 1, 1, 1, 1, 1, 2, 1, 2]",1min,False,1,1,run_000001,1000001
2,DE,AB,Detached,DE.04,2129.2,20,average,False,3,"[2, 2, 1, 1, 1, 2, 1, 2, 2, 1, 2, 2, 2, 1, 4, ...",1min,False,1,2,run_000002,1000002
3,DE,AB,Detached,DE.05,1878.4,18,average,False,15,"[1, 2, 1, 2, 2, 1, 5, 1, 1, 3, 1, 5, 3, 1, 2, ...",1min,False,1,3,run_000003,1000003
4,DE,AB,Detached,DE.06,1409.3,17,average,False,10,"[3, 2, 2, 1, 1, 1, 1, 3, 1, 2, 2, 2, 1, 5, 1, ...",1min,False,1,4,run_000004,1000004
5,DE,SFH,Detached,DE.01,34.7,1,average,False,8,[1],1min,False,1,5,run_000005,1000005
6,DE,SFH,Detached,DE.02,95.6,1,average,False,7,[2],1min,False,1,6,run_000006,1000006
7,DE,SFH,Detached,DE.03,77.9,1,average,False,8,[1],1min,False,1,7,run_000007,1000007
8,DE,SFH,Detached,DE.04,36.8,1,average,False,15,[1],1min,False,1,8,run_000008,1000008
9,DE,SFH,Detached,DE.05,132.7,1,average,False,13,[5],1min,False,1,9,run_000009,1000009


In [9]:
pd.read_parquet("manifest.parquet")

,country,buildingType,surrounding,buildingAgeBin,a_ref,n_apartments,year_type,future,climateRegion,n_persons,freq,hasFirePlace,cores,scenario_id,run_id,seed
0,DE,AB,Terraced,DE.02,1938.3,20,average,False,9,"[2, 2, 3, 2, 2, 2, 2, 1, 2, 2, 3, 2, 2, 4, 1, ...",1min,False,1,0,run_000000,1000000
1,DE,AB,Semi,DE.03,1363.8,16,average,False,14,"[1, 3, 2, 1, 3, 2, 1, 2, 1, 1, 1, 1, 1, 2, 1, 2]",1min,False,1,1,run_000001,1000001
2,DE,AB,Detached,DE.04,2129.2,20,average,False,3,"[2, 2, 1, 1, 1, 2, 1, 2, 2, 1, 2, 2, 2, 1, 4, ...",1min,False,1,2,run_000002,1000002
3,DE,AB,Detached,DE.05,1878.4,18,average,False,15,"[1, 2, 1, 2, 2, 1, 5, 1, 1, 3, 1, 5, 3, 1, 2, ...",1min,False,1,3,run_000003,1000003
4,DE,AB,Detached,DE.06,1409.3,17,average,False,10,"[3, 2, 2, 1, 1, 1, 1, 3, 1, 2, 2, 2, 1, 5, 1, ...",1min,False,1,4,run_000004,1000004
5,DE,SFH,Detached,DE.01,34.7,1,average,False,8,[1],1min,False,1,5,run_000005,1000005
6,DE,SFH,Detached,DE.02,95.6,1,average,False,7,[2],1min,False,1,6,run_000006,1000006
7,DE,SFH,Detached,DE.03,77.9,1,average,False,8,[1],1min,False,1,7,run_000007,1000007
8,DE,SFH,Detached,DE.04,36.8,1,average,False,15,[1],1min,False,1,8,run_000008,1000008
9,DE,SFH,Detached,DE.05,132.7,1,average,False,13,[5],1min,False,1,9,run_000009,1000009


# Generator

In [14]:
import pandas as pd

path = r"C:\Users\Samuel\FH_Aachen\tsib_simuflex_fork\tsib_simuflex\Generation\results\run_000047.h5"

# see what's actually stored inside, in case you forget the exact keys
with pd.HDFStore(path, mode="r") as store:
    print(store.keys())   # -> ['/timeseries', '/params']

timeseries = pd.read_hdf(path, key="timeseries")
params = pd.read_hdf(path, key="params")

print(params.T)          # transpose — easier to read a one-row table this way
print(timeseries.head())
print(timeseries.dtypes)

['/params', '/timeseries']
                                                                        0
country                                                                DE
buildingType                                                          SFH
surrounding                                                      Detached
buildingAgeBin                                                      DE.12
a_ref                                                               115.8
n_apartments                                                            1
year_type                                                         average
future                                                              False
climateRegion                                                           4
n_persons                                                             [3]
freq                                                                 1min
hasFirePlace                                                        False
cores      

In [32]:
store = pd.HDFStore(r"C:\Users\Samuel\Downloads\Ziel\test.h5", mode="w")

In [ ]:
os.replace()

In [38]:
store.put("manifest", df, format="table")

In [39]:
store.close()

In [41]:
store.open()

In [42]:
store.select("manifest")

,Unnamed: 0,Code_BuildingVariant,Date_Entry,Code_StatusDataset,Code_Country,Code_Building,Description_BuildingVariant,Description_BuildingVariant_National,Code_BuildingType,Code_DataType_Building,...,Q_Sol_South,Q_Sol_West,Q_Sol_North,q_sol,q_int,tau,a_H,gamma_h_gn,eta_h_gn,q_h_nd
0,11,AT.N.AB.01.Gen.ReEx.001.001,2015-09-04 00:00:00,Typology,AT,AT.N.AB.01.Gen.ReEx.001,Existing condition,Zustand Bestand,AT.N.AB.01.Gen,ReEx,...,0.000000,6113.728530,0.000000,8.975227,15.264,16.882835,1.362761,0.121795,0.949819,175.994029
1,14,AT.N.AB.02.Gen.ReEx.001.001,2010-09-07 00:00:00,Typology,AT,AT.N.AB.02.Gen.ReEx.001,Existing condition,Zustand Bestand,AT.N.AB.02.Gen,ReEx,...,111.334041,4263.592788,41.090868,8.083775,15.264,13.099762,1.236659,0.093664,0.951285,227.060285
2,17,AT.N.AB.03.Gen.ReEx.001.001,2010-09-07 00:00:00,Typology,AT,AT.N.AB.03.Gen.ReEx.001,Existing condition,Zustand Bestand,AT.N.AB.03.Gen,ReEx,...,5296.741632,355.400136,2335.270392,9.980548,15.264,15.986860,1.332895,0.120774,0.947081,185.113920
3,20,AT.N.AB.04.Gen.ReEx.001.001,2010-09-07 00:00:00,Typology,AT,AT.N.AB.04.Gen.ReEx.001,Existing condition,Zustand Bestand,AT.N.AB.04.Gen,ReEx,...,11725.543557,265.341258,2134.543320,15.381327,15.264,15.670092,1.322336,0.144009,0.933254,184.201843
4,23,AT.N.AB.05.Gen.ReEx.001.001,2010-09-07 00:00:00,Typology,AT,AT.N.AB.05.Gen.ReEx.001,Existing condition,Zustand Bestand,AT.N.AB.05.Gen,ReEx,...,5842.574010,469.031472,3886.177932,7.645423,15.264,45.440649,2.314688,0.281850,0.961117,59.263580
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
831,2979,SI.N.MFH-AB.03-04.Gen.SyAv.001.001,2015-07-03 00:00:00,Analysis,SI,SI.N.MFH-AB.03-04.Gen.SyAv.001,0,0,SI.N.MFH-AB.03-04.Gen,SyAv,...,9069.457950,6771.033675,3043.859175,8.547734,14.832,23.406668,1.580222,0.167339,0.950122,117.501205
832,2980,SI.N.MFH-AB.05-06.Gen.SyAv.001.001,2015-07-03 00:00:00,Analysis,SI,SI.N.MFH-AB.05-06.Gen.SyAv.001,0,0,SI.N.MFH-AB.05-06.Gen,SyAv,...,14004.334417,10455.290764,4700.084839,6.697053,14.832,52.073181,2.535773,0.320371,0.961403,46.502309
833,2981,SI.N.SFH-TH.01-02.Gen.SyAv.001.001,2015-07-03 00:00:00,Analysis,SI,SI.N.SFH-TH.01-02.Gen.SyAv.001,0,0,SI.N.SFH-TH.01-02.Gen,SyAv,...,649.838700,485.153550,218.096550,12.847136,14.832,12.861238,1.228708,0.122245,0.933030,200.597076
834,2982,SI.N.SFH-TH.03-04.Gen.SyAv.001.001,2015-07-03 00:00:00,Analysis,SI,SI.N.SFH-TH.03-04.Gen.SyAv.001,0,0,SI.N.SFH-TH.03-04.Gen,SyAv,...,411.943927,307.547179,138.255154,7.930485,14.832,22.540807,1.551360,0.166208,0.947942,115.374672


In [38]:
episcope_combis(r"C:\Users\Samuel\FH_Aachen\tsib_simuflex_fork\tsib_simuflex\tsib\data\episcope\episcope.csv")

,Code_BuildingVariant,Code_BuildingSizeClass,Code_AttachedNeighbours,Code_ConstructionYearClass
187,DE.N.TH.02.Gen.ReEx.001.001,TH,B_N2,DE.02
188,DE.N.TH.03.Gen.ReEx.001.001,TH,B_N2,DE.03
189,DE.N.TH.04.Gen.ReEx.001.001,TH,B_N1,DE.04
190,DE.N.TH.05.Gen.ReEx.001.001,TH,B_N2,DE.05
191,DE.N.TH.06.Gen.ReEx.001.001,TH,B_N2,DE.06
192,DE.N.TH.07.Gen.ReEx.001.001,TH,B_N2,DE.07
193,DE.N.TH.08.Gen.ReEx.001.001,TH,B_N1,DE.08
194,DE.N.TH.09.Gen.ReEx.001.001,TH,B_N2,DE.09
195,DE.N.TH.10.Gen.ReEx.001.001,TH,B_N1,DE.10
196,DE.N.TH.11.Gen.ReEx.001.001,TH,B_N1,DE.11


In [44]:
df_new = df[(df["Code_Country"] == "DE") &(df["Code_BuildingVariant"].str.contains("SFH"))]

In [49]:
df_new.loc[175:179]

,Unnamed: 0,Code_BuildingVariant,Date_Entry,Code_StatusDataset,Code_Country,Code_Building,Description_BuildingVariant,Description_BuildingVariant_National,Code_BuildingType,Code_DataType_Building,...,Q_Sol_South,Q_Sol_West,Q_Sol_North,q_sol,q_int,tau,a_H,gamma_h_gn,eta_h_gn,q_h_nd
175,545,DE.N.SFH.02.Gen.ReEx.001.001,2010-09-07 00:00:00,Typology,DE,DE.N.SFH.02.Gen.ReEx.001,0,0,DE.N.SFH.02.Gen,ReEx,...,625.51440,481.46805,48.818700,12.128378,15.552,9.720661,1.124022,0.093622,0.936326,269.742045
176,548,DE.N.SFH.03.Gen.ReEx.001.001,2010-09-07 00:00:00,Typology,DE,DE.N.SFH.03.Gen.ReEx.001,0,0,DE.N.SFH.03.Gen,ReEx,...,2457.37800,572.72670,423.676575,13.634850,15.552,12.078144,1.202605,0.118778,0.931395,218.542445
177,551,DE.N.SFH.04.Gen.ReEx.001.001,2010-09-07 00:00:00,Typology,DE,DE.N.SFH.04.Gen.ReEx.001,0,0,DE.N.SFH.04.Gen,ReEx,...,960.61140,204.54525,115.072650,13.679449,15.552,9.576125,1.119204,0.097398,0.932917,272.851886
178,554,DE.N.SFH.05.Gen.ReEx.001.001,2010-09-07 00:00:00,Typology,DE,DE.N.SFH.05.Gen.ReEx.001,0,0,DE.N.SFH.05.Gen,ReEx,...,703.70370,560.13930,142.969050,15.071433,15.552,9.783928,1.126131,0.104251,0.929209,265.292457
179,557,DE.N.SFH.06.Gen.ReEx.001.001,2010-09-07 00:00:00,Typology,DE,DE.N.SFH.06.Gen.ReEx.001,0,0,DE.N.SFH.06.Gen,ReEx,...,1858.67136,314.68500,263.969685,16.195533,15.552,13.411644,1.247055,0.141350,0.924217,195.260705


In [45]:
df_new[["Code_BuildingVariant", 'Code_AttachedNeighbours', "Code_StatusDataset", "Code_Country"]]

,Code_BuildingVariant,Code_AttachedNeighbours,Code_StatusDataset,Code_Country
174,DE.N.SFH.01.Gen.ReEx.001.001,B_Alone,Typology,DE
175,DE.N.SFH.02.Gen.ReEx.001.001,B_Alone,Typology,DE
176,DE.N.SFH.03.Gen.ReEx.001.001,B_Alone,Typology,DE
177,DE.N.SFH.04.Gen.ReEx.001.001,B_Alone,Typology,DE
178,DE.N.SFH.05.Gen.ReEx.001.001,B_Alone,Typology,DE
179,DE.N.SFH.06.Gen.ReEx.001.001,B_Alone,Typology,DE
180,DE.N.SFH.06.LightFrame.ReEx.001.001,B_Alone,Typology,DE
181,DE.N.SFH.07.Gen.ReEx.001.001,B_N1,Typology,DE
182,DE.N.SFH.08.Gen.ReEx.001.001,B_N1,Typology,DE
183,DE.N.SFH.09.Gen.ReEx.001.001,B_Alone,Typology,DE
